## GNBF_6_grid2stlwskirt

**Input:** npz-formatted surface file (like Segments1_retrieved.npz) \
**Output**: stl-formatted surface file (like Segments1_retrievedwskirt.stl)

Code at the end, Curvature analysis, returns the 'C" in a best-fit to 

$$
z = A + Bx + {1\over2}Cx^2 
$$

which is really only meaningful for smooth surfaces.

In [8]:
# Generic imports
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import matplotlib.pyplot as plt
import copy
import os

# Local imports
import sys
sys.path.append('../../../../')
import stlstuff as sls
import imagestuff as ims

In [10]:
# Graphics
%matplotlib widget

In [12]:
# Get the retrieved surface
pwd = os.getcwd()
print(pwd)
startpos = pwd.find('crystals/')
case = pwd[startpos+9:]; print(case)
Segmentnamelistroot = 'Segments1'

surfaceroot = ''
npzfilename = Segmentnamelistroot+'_retrieved.npz'
npzfile = np.load(npzfilename); print('input: '+npzfilename)
stlfilename = Segmentnamelistroot+'_retrievedwskirt.stl'
stlfile = stlfilename; print('output: '+stlfile)

imageroot = np.array_str(npzfile['imageroot'])

dx,dy,cA,cB,cC,cD,Filename = ims.getc2(surfaceroot, 'SEMimages/', imageroot)
nx1list = npzfile['nx1list']
nx2list = npzfile['nx2list']
ny1list = npzfile['ny1list']
ny2list = npzfile['ny2list']

nxi = npzfile['nxi']; #print('this is nxi', nxi)
nyi = npzfile['nyi']; #print('this is nyi', nyi)
dnx = npzfile['dnx']; #print('this is dnx', dnx)
dny = npzfile['dny']; #print('this is dny', dny)
solution = npzfile['solution']; print('shape of the solution is ', np.shape(solution))
# print(solution[:,0])
# print(solution[0,:])
# print (dx,dy)

/Users/nesh/Documents/Repositories/ice2021/crystals/2023-07-20/50pa/case3.0 (calibration)
2023-07-20/50pa/case3.0 (calibration)
input: Segments1_retrieved.npz
output: Segments1_retrievedwskirt.stl
SEMimages/case3.0-A.bmp
SEMimages/case3.0-B.bmp
SEMimages/case3.0-C.bmp
SEMimages/case3.0-D.bmp
shape of the solution is  (960, 1280)


In [14]:
# Extract the surface
nx1tot = min(nx1list)
nx2tot = max(nx2list)
ny1tot = min(ny1list)
ny2tot = max(ny2list)
nxtot = nx2tot-nx1tot; print(nxtot)
nytot = ny2tot-ny1tot; print(nytot)
# settemp = solution[ny1tot:ny2tot,nx1tot:nx2tot]; print (np.shape(settemp))
settemp = solution[2]; print (np.shape(settemp))

40
160
(1280,)


The next cell adds a skirt around the data

In [17]:
# get the dimensions 
nx,ny = np.shape(settemp)

# define the bottom of the skirt
bottom = np.min(settemp)- 20


# add a row to beginning 
startrow = np.ones(ny)*bottom; #print(np.shape(startrow))

# add a row to the end
settemp1 = np.vstack ((startrow,settemp,startrow)); #print(np.shape(settemp1))
settemp2 = np.transpose(settemp1); #print(np.shape(settemp2))

# add a column to beginning
newcol = np.ones(nx+2)*bottom; #print(np.shape(newcol))

# add a column to the end
settemp3 = np.vstack ((newcol,settemp2,newcol))
settemp4 = np.transpose (settemp3)

# Update the dimensions
nytot, nxtot = np.shape(settemp4)
print (nxtot,nytot)

ValueError: not enough values to unpack (expected 2, got 1)

In [ ]:
# Display the surface here


# Set display parameters
fontsize = 25
linewidth = 2
xmaxtot = (nxtot-1)*dx; print (xmaxtot)
ymaxtot = (nytot-1)*dy; print (ymaxtot)
surf_ytot = np.linspace(0,ymaxtot,nytot); #print surf_ytot[1]-surf_ytot[0]; 
surf_xtot = np.linspace(0,xmaxtot,nxtot); #print surf_xtot[1]-surf_xtot[0]; 
surf_xgridtot, surf_ygridtot = np.meshgrid(surf_xtot,surf_ytot)
surf_ygridtot_flipped = np.flipud(surf_ygridtot)

# Display it
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(surf_xgridtot, surf_ygridtot_flipped, settemp4, rstride=1,cstride=1, color='g')
ax.view_init(azim=-100,elev=22)
ax.set_xlabel('x',fontsize=fontsize)
ax.set_ylabel('y',fontsize=fontsize)
ax.set_zlabel('z',fontsize=fontsize)

In [ ]:
# Save as stl (with possible vertical amplification)
scale = 1/dx # sometimes useful as a vertical scale
scale = 5 # for an explicit vertical scale
sls.numpy2stl(settemp4, stlfile, scale=scale, solid=False)
print('Complete')

In [ ]:
# Curvature analysis
# nx and ny are swapped above here
print('shape of settemp', np.shape(settemp))

# Through the middle of x, a line in the y-direction
iymid = round(nx/2); print('iymid',iymid)
z_of_x = settemp[iymid,:]; print('z_of_x',len(z_of_x))
x_mid = surf_xtot[1:-1]
curvature_in_x = sls.curvature(x_mid,z_of_x)
print('curvature_in_x',curvature_in_x)

# Through the middle of x, a line in the y-direction
ixmid = round(ny/2); print('ixmid',ixmid)
z_of_y = settemp[:,ixmid]; print('z_of_y',len(z_of_y))
y_mid = surf_ytot[1:-1]
curvature_in_x = sls.curvature(y_mid,z_of_y)
print('curvature_in_x',curvature_in_x)